<div align="center">
<a href="https://rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/RapidFire - Blue bug -white text.svg" width="115"></a>
<a href="https://discord.gg/6vSTtncKNN"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/discord-button.svg" width="145"></a>
<a href="https://oss-docs.rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/documentation-button.svg" width="125"></a>
<br/>
Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/RapidFireAI/rapidfireai">GitHub</a></i> ⭐
</div>

## RapidFire AI + Optuna: Adaptive RAG Hyperparameter Search on FiQA

This tutorial shows how to use **RFOptuna** for Bayesian hyperparameter optimization
of a RAG (Retrieval-Augmented Generation) pipeline on the **FiQA** financial Q&A dataset.

**Key difference from RFGridSearch / RFRandomSearch:**
- Grid/Random search decide all configs upfront
- **RFOptuna** uses Optuna's TPE sampler to suggest initial configs, then **prunes underperforming pipelines after each shard** and replaces them with smarter suggestions

**What this notebook demonstrates:**
- Defining a RAG search space with `List(...)` over retrieval parameters (chunk size, reranker top-n)
- Using `RFOptuna` in **evals mode** to adaptively search the space
- Optuna-driven pruning of weak retrieval configs mid-evaluation
- Inspecting the Optuna study to find the best RAG configuration

### Enable Metric Loggers

In [ ]:
import os
os.environ["RF_MLFLOW_ENABLED"] = "true"
os.environ["RF_TENSORBOARD_ENABLED"] = "true"
os.environ["RF_TRACKIO_ENABLED"] = "true"

### Imports

Note: `RFOptuna` requires the `optuna` package. Install with:
```bash
pip install optuna
```

In [ ]:
from rapidfireai import Experiment
from rapidfireai.automl import (
    List,
    RFOptuna,
    RFLangChainRagSpec,
    RFvLLMModelConfig,
    RFPromptManager,
)
import math
import pandas as pd
from pathlib import Path
from typing import List as listtype, Dict, Any

### Load FiQA Dataset and Relevance Labels

In [ ]:
from datasets import load_dataset

dataset_dir = Path("datasets")

fiqa_dataset = load_dataset(
    "json", data_files=str(dataset_dir / "fiqa" / "queries.jsonl"), split="train"
).select(range(256))
fiqa_dataset = fiqa_dataset.rename_columns({"text": "query", "_id": "query_id"})

qrels = pd.read_csv(str(dataset_dir / "fiqa" / "qrels.tsv"), sep="\t")
qrels = qrels.rename(
    columns={"query-id": "query_id", "corpus-id": "corpus_id", "score": "relevance"}
)

### Create Experiment

In [ ]:
experiment = Experiment(experiment_name="exp-optuna-rag-fiqa", mode="evals")

### Define RAG Pipeline with Search Space

Instead of listing every combination (GridSearch) or sampling blindly (RandomSearch),
we define a **search space** using `List(...)` inside the RAG spec.
Optuna's TPE sampler will intelligently explore this space.

**What Optuna controls:**
- `text_splitter`: Chunk size (128 vs 256 tokens)
- `reranker_cfg.top_n`: Number of documents after reranking (2 vs 5)

**What stays fixed:**
- Embedding model (`all-MiniLM-L6-v2`), FAISS vector store, similarity search with k=15

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, JSONLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

batch_size = 128

rag_spec = RFLangChainRagSpec(
    document_loader=DirectoryLoader(
        path=str(dataset_dir / "fiqa"),
        glob="corpus.jsonl",
        loader_cls=JSONLoader,
        loader_kwargs={
            "jq_schema": ".",
            "content_key": "text",
            "metadata_func": lambda record, metadata: {
                "corpus_id": int(record.get("_id"))
            },
            "json_lines": True,
            "text_content": False,
        },
        sample_seed=42,
    ),
    text_splitter=List([
        RecursiveCharacterTextSplitter.from_tiktoken_encoder(
            encoding_name="gpt2", chunk_size=256, chunk_overlap=32
        ),
        RecursiveCharacterTextSplitter.from_tiktoken_encoder(
            encoding_name="gpt2", chunk_size=128, chunk_overlap=32
        ),
    ]),
    embedding_cfg={
        "class": HuggingFaceEmbeddings,
        "model_name": "sentence-transformers/all-MiniLM-L6-v2",
        "model_kwargs": {"device": "cuda:0"},
        "encode_kwargs": {"normalize_embeddings": True, "batch_size": batch_size},
    },
    vector_store_cfg={
        "type": "faiss",
    },
    search_cfg={
        "type": "similarity",
        "k": 15,
    },
    reranker_cfg={
        "class": CrossEncoderReranker,
        "model_name": "cross-encoder/ms-marco-MiniLM-L6-v2",
        "model_kwargs": {"device": "cuda:0"},
        "top_n": List([2, 5]),
    },
    enable_gpu_search=True,
)

### Define Data Processing Functions

In [ ]:
def sample_preprocess_fn(
    batch: Dict[str, listtype], rag: RFLangChainRagSpec, prompt_manager: RFPromptManager
) -> Dict[str, listtype]:
    """Prepare the final inputs given to the generator model."""
    INSTRUCTIONS = (
        "Utilize your financial knowledge, give your answer or opinion "
        "to the input question or subject matter."
    )

    all_context = rag.get_context(batch_queries=batch["query"], serialize=False)

    retrieved_documents = [
        [doc.metadata["corpus_id"] for doc in docs] for docs in all_context
    ]

    serialized_context = rag.serialize_documents(all_context)
    batch["query_id"] = [int(query_id) for query_id in batch["query_id"]]

    return {
        "prompts": [
            [
                {"role": "system", "content": INSTRUCTIONS},
                {
                    "role": "user",
                    "content": (
                        f"Here is some relevant context:\n{context}. "
                        f"\nNow answer the following question using the "
                        f"context provided earlier:\n{question}"
                    ),
                },
            ]
            for question, context in zip(batch["query"], serialized_context)
        ],
        "retrieved_documents": retrieved_documents,
        **batch,
    }


def sample_postprocess_fn(batch: Dict[str, listtype]) -> Dict[str, listtype]:
    """Attach ground truth documents for metric computation."""
    batch["ground_truth_documents"] = [
        qrels[qrels["query_id"] == query_id]["corpus_id"].tolist()
        for query_id in batch["query_id"]
    ]
    return batch

### Define Custom RAG Eval Metrics

In [ ]:
def compute_ndcg_at_k(retrieved_docs: set, expected_docs: set, k=5):
    """Compute NDCG@k for a single query."""
    relevance = [1 if doc in expected_docs else 0 for doc in list(retrieved_docs)[:k]]
    dcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(relevance))
    ideal_length = min(k, len(expected_docs))
    ideal_relevance = [3] * ideal_length + [0] * (k - ideal_length)
    idcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(ideal_relevance))
    return dcg / idcg if idcg > 0 else 0.0


def compute_rr(retrieved_docs: set, expected_docs: set):
    """Compute Reciprocal Rank (RR) for a single query."""
    for i, retrieved_doc in enumerate(retrieved_docs):
        if retrieved_doc in expected_docs:
            return 1 / (i + 1)
    return 0


def sample_compute_metrics_fn(batch: Dict[str, listtype]) -> Dict[str, Dict[str, Any]]:
    """Compute retrieval metrics per batch."""
    precisions, recalls, f1_scores, ndcgs, rrs = [], [], [], [], []
    total_queries = len(batch["query"])

    for pred, gt in zip(batch["retrieved_documents"], batch["ground_truth_documents"]):
        expected_set = set(gt)
        retrieved_set = set(pred)

        true_positives = len(expected_set.intersection(retrieved_set))
        precision = true_positives / len(retrieved_set) if len(retrieved_set) > 0 else 0
        recall = true_positives / len(expected_set) if len(expected_set) > 0 else 0
        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall) > 0
            else 0
        )

        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)
        ndcgs.append(compute_ndcg_at_k(retrieved_set, expected_set, k=5))
        rrs.append(compute_rr(retrieved_set, expected_set))

    return {
        "Total": {"value": total_queries},
        "Precision": {"value": sum(precisions) / total_queries},
        "Recall": {"value": sum(recalls) / total_queries},
        "F1 Score": {"value": sum(f1_scores) / total_queries},
        "NDCG@5": {"value": sum(ndcgs) / total_queries},
        "MRR": {"value": sum(rrs) / total_queries},
    }


def sample_accumulate_metrics_fn(
    aggregated_metrics: Dict[str, listtype],
) -> Dict[str, Dict[str, Any]]:
    """Accumulate metrics across all batches."""
    num_queries_per_batch = [m.get("value", 0) for m in aggregated_metrics.get("Total", [])]
    total_queries = sum(num_queries_per_batch)
    algebraic_metrics = ["Precision", "Recall", "F1 Score", "NDCG@5", "MRR"]

    return {
        "Total": {"value": total_queries},
        **{
            metric: {
                "value": sum(
                    m["value"] * queries
                    for m, queries in zip(
                        aggregated_metrics[metric], num_queries_per_batch
                    )
                )
                / total_queries,
                "is_algebraic": True,
                "value_range": (0, 1),
            }
            for metric in algebraic_metrics
        },
    }

### Define vLLM Generator Config with RAG

We use a single generator model (Qwen2.5-0.5B-Instruct) and let Optuna
search over the **retrieval parameters** defined in the RAG spec above.

In [ ]:
vllm_config = RFvLLMModelConfig(
    model_config={
        "model": "Qwen/Qwen2.5-0.5B-Instruct",
        "dtype": "half",
        "gpu_memory_utilization": 0.7,
        "tensor_parallel_size": 1,
        "distributed_executor_backend": "mp",
        "enable_chunked_prefill": False,
        "enable_prefix_caching": True,
        "max_model_len": 4096,
        "disable_log_stats": True,
    },
    sampling_params={
        "temperature": 0.8,
        "top_p": 0.95,
        "max_tokens": 512,
    },
    rag=rag_spec,
    prompt_manager=None,
)

config_template = {
    "vllm_config": vllm_config,
    "batch_size": batch_size,
    "preprocess_fn": sample_preprocess_fn,
    "postprocess_fn": sample_postprocess_fn,
    "compute_metrics_fn": sample_compute_metrics_fn,
    "accumulate_metrics_fn": sample_accumulate_metrics_fn,
    "online_strategy_kwargs": {
        "strategy_name": "normal",
        "confidence_level": 0.95,
        "use_fpc": True,
    },
}

### Create RFOptuna Config Group

Key parameters:
- **`n_initial=4`**: Start with 4 pipeline configs (sampled by TPE)
- **`budget=6`**: Allow up to 6 total configs (including replacements for pruned pipelines)
- **`objective="maximize:NDCG@5"`**: Optuna maximizes NDCG@5 to decide pruning
- **`sampler="tpe"`**: Tree-structured Parzen Estimator (Bayesian)
- **`pruner="median"`**: Prune pipelines performing worse than the median at each shard

Note: `trainer_type` is `None` for evals mode (RAG). The search space comes from
`List(...)` values embedded in the `rag_spec` and `vllm_config`.

In [ ]:
config_group = RFOptuna(
    configs=[config_template],
    trainer_type=None,
    n_initial=4,
    budget=6,
    objective="maximize:NDCG@5",
    sampler="tpe",
    pruner="median",
    seed=42,
)

### Run Optuna-Powered RAG Evaluation

Behind the scenes:
1. `RFOptuna.get_runs()` asks Optuna's TPE sampler for 4 initial RAG configs
2. RapidFire evaluates all 4 concurrently using shard-based scheduling
3. After each shard completes, the Optuna callback:
   - Reports the pipeline's `NDCG@5` to Optuna
   - Checks if Optuna's median pruner wants to prune the pipeline
   - If pruned, suggests a replacement config (up to budget of 6)
4. RapidFire stops pruned pipelines and starts replacements automatically

In [ ]:
results = experiment.run_evals(
    config_group=config_group,
    dataset=fiqa_dataset,
    num_shards=4,
    seed=42,
)

### Inspect Optuna Study Results

After evaluation, the Optuna study object is accessible on the `RFOptuna` instance.
You can inspect which trials were completed, pruned, or failed.

In [ ]:
from optuna.trial import TrialState

study = config_group._study

print(f"Number of trials: {len(study.trials)}")
done = [t for t in study.trials if t.state == TrialState.COMPLETE]
print(f"Completed trials: {len(done)}")
try:
    bt = study.best_trial
    print(f"Best trial value (NDCG@5): {bt.value:.4f}")
    print(f"Best trial params: {bt.params}")
except (RuntimeError, ValueError) as exc:
    print(f"No best trial available yet ({exc}). Re-run evals or check logs.")
print()

for trial in study.trials:
    print(
        f"  Trial {trial.number}: state={trial.state.name}, "
        f"params={trial.params}, value={trial.value}"
    )

### View Results

In [ ]:
results_df = pd.DataFrame([
    {
        k: v["value"] if isinstance(v, dict) and "value" in v else v
        for k, v in {**metrics_dict, "run_id": run_id}.items()
    }
    for run_id, (_, metrics_dict) in results.items()
])

results_df

### End Experiment

In [ ]:
experiment.end()

---

### Comparison: RFOptuna vs RFGridSearch vs RFRandomSearch for RAG

| Feature | RFGridSearch | RFRandomSearch | RFOptuna |
|---|---|---|---|
| Config selection | All combos upfront | Random sample upfront | Bayesian (TPE) — learns from results |
| Pruning | Manual via IC Ops | Manual via IC Ops | Automatic (Median / Hyperband pruner) |
| Replacement | Manual clone-modify | Manual clone-modify | Automatic — new suggestions within budget |
| Search space | `List([...])` | `List([...])`, `Range(...)` | `List([...])`, `Range(...)` |
| Best for | Small discrete spaces | Large spaces, no adaptation | Large spaces, adaptive exploration |
| RAG use case | Few chunk/reranker combos | Many retrieval variants | Optimizing retrieval quality adaptively |

<div align="center">
<a href="https://rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/RapidFire - Blue bug -white text.svg" width="115"></a>
<a href="https://discord.gg/6vSTtncKNN"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/discord-button.svg" width="145"></a>
<a href="https://oss-docs.rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/documentation-button.svg" width="125"></a>
<br/>
Thanks for trying RapidFire AI! ⭐ <i>Star us on <a href="https://github.com/RapidFireAI/rapidfireai">GitHub</a></i> ⭐
</div>